# 07 — Agent Prototyping

This notebook tests and validates the OpsAgent LangGraph investigation agent.

**Sections:**
1. Setup & Imports
2. Individual Tool Testing
3. Graph Compilation & Visualization
4. AgentState Construction
5. Routing Logic Verification
6. Full Investigation (End-to-End)
7. Report Inspection

**Requirements:**
- Docker infrastructure stack running (`make infra-up && make demo-up`)
- `GEMINI_API_KEY` set in `.env`
- Runbooks indexed into ChromaDB

In [ ]:
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

# Ensure project root is on the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv

load_dotenv(project_root / ".env")

print(f"Project root: {project_root}")
print(f"GEMINI_API_KEY set: {'GEMINI_API_KEY' in os.environ}")

## 1. Individual Tool Testing

Test each of the 5 agent tools independently.

In [ ]:
# Tool 1: get_topology — static graph, no external dependencies
from src.agent.tools.get_topology import get_topology

# Full topology
full_topo = get_topology.invoke({"service_name": None})
print(f"Full topology: {len(full_topo['nodes'])} nodes, {len(full_topo['edges'])} edges")
for n in full_topo['nodes']:
    print(f"  - {n['name']}")

# Subgraph for cartservice
sub = get_topology.invoke({"service_name": "cartservice"})
print("\ncartservice subgraph:")
print(f"  Upstream (root cause suspects): {sub['upstream']}")
print(f"  Downstream (symptom services):  {sub['downstream']}")

In [ ]:
# Tool 2: query_metrics — requires Prometheus running
from src.agent.tools.query_metrics import query_metrics

result = query_metrics.invoke({
    "service_name": "frontend",
    "metric_name": "cpu_usage",
    "time_range_minutes": 5,
})
print("CPU usage for frontend:")
print(f"  Data points: {len(result.get('values', []))}")
if result.get('stats'):
    print(f"  Stats: {result['stats']}")
    print(f"  Anomalous: {result['anomalous']}")
else:
    print(f"  Note: {result.get('note', result.get('error', 'No data'))}")

In [ ]:
# Tool 3: search_logs — requires Loki running
from src.agent.tools.search_logs import search_logs

result = search_logs.invoke({
    "query": "error",
    "service_filter": "cartservice",
    "time_range_minutes": 60,
})
print("Log search for 'error' in cartservice:")
print(f"  Total entries: {result['total_count']}")
print(f"  Error entries: {result['error_count']}")
if result['entries']:
    print(f"  First entry: {result['entries'][0]['message'][:100]}")

In [ ]:
# Tool 4: search_runbooks — requires ChromaDB populated
from src.agent.tools.search_runbooks import search_runbooks

result = search_runbooks.invoke({
    "query": "Redis connection pool exhaustion",
    "top_k": 3,
})
print(f"Runbook search results: {len(result['results'])} matches")
for r in result['results']:
    print(f"  [{r['relevance_score']:.2f}] {r['title']}")
    print(f"    {r['content'][:100]}...")

## 2. Graph Compilation & Visualization

In [ ]:
from src.agent.graph import build_graph

graph = build_graph()
print(f"Graph type: {type(graph).__name__}")
print("Graph compiled successfully!")

# Try to get the Mermaid diagram
try:
    mermaid = graph.get_graph().draw_mermaid()
    print(f"\nGraph structure (Mermaid):\n{mermaid}")
except Exception as e:
    print(f"Mermaid rendering not available: {e}")

## 3. Routing Logic Verification

In [ ]:
from src.agent.graph import should_continue

# Test routing decisions
test_cases = [
    {"root_cause_confidence": 0.85, "tool_calls_remaining": 5},
    {"root_cause_confidence": 0.3,  "tool_calls_remaining": 0},
    {"root_cause_confidence": 0.4,  "tool_calls_remaining": 5},
    {"root_cause_confidence": 0.7,  "tool_calls_remaining": 3},
]

print("Routing logic test:")
for tc in test_cases:
    decision = should_continue(tc)
    print(f"  confidence={tc['root_cause_confidence']}, "
          f"remaining={tc['tool_calls_remaining']} → {decision}")

## 4. AgentExecutor Setup

In [ ]:
from src.agent.executor import AgentExecutor

# Load from config file
config_path = str(project_root / "configs" / "agent_config.yaml")
executor = AgentExecutor.from_config(config_path)
inv = executor.config["agent"]["investigation"]
print("AgentExecutor initialized")
print(f"  Max tool calls: {inv['max_tool_calls']}")
print(f"  Confidence threshold: {inv['confidence_threshold']}")

## 5. Full Investigation (End-to-End)

Run a complete investigation with a synthetic alert.

**Note:** This cell requires `GEMINI_API_KEY` and the Docker stack running.

In [ ]:
# Construct a synthetic alert
alert = {
    "title": "LSTM-AE Anomaly Detected — High CPU in cartservice",
    "severity": "high",
    "timestamp": datetime.now(UTC).isoformat(),
    "affected_services": ["cartservice", "checkoutservice"],
    "anomaly_score": 0.45,
    "threshold": 0.253,
}

print("Starting investigation...")
print(f"Alert: {alert['title']}")
print(f"Affected services: {alert['affected_services']}")
print()

# Run the investigation
result = executor.investigate(alert=alert)

print(f"\n{'='*60}")
print("Investigation complete!")
print(f"{'='*60}")
print(f"Root cause: {result['root_cause']}")
print(f"Confidence: {result['root_cause_confidence']:.0%}")
print(f"Top 3 predictions: {result['top_3_predictions']}")
print(f"Recommended actions: {result['recommended_actions']}")

## 6. Report Inspection

In [ ]:
# Display the full RCA report
if result.get('rca_report'):
    print(result['rca_report'])
else:
    print("No RCA report generated.")

## 7. Summary

| Component | Status |
|-----------|--------|
| AgentState TypedDict | 13 fields, add_messages reducer |
| 5 Agent Tools | get_topology, query_metrics, search_logs, search_runbooks, discover_causation |
| LangGraph Workflow | 5 nodes + conditional routing |
| AgentExecutor | Dual-mode (live + offline) |
| Unit Tests | 62 new tests, all passing |
| RCA Report | Structured template with evidence chain |